# ☀️ Solar Filament Segmentation 2026: Baseline Pipeline (U-Net)

Welcome to the **Baseline Pipeline** for the Solar Filament Segmentation Challenge! 

### 📌 Overview & Architecture
This notebook provides a complete, start-to-finish PyTorch pipeline for solar filament segmentation:
- **Dataset Customization:** On-the-fly binary mask generation from JSON annotations.
- **Augmentations:** Lightweight spatial transforms via `albumentations`.
- **Model Architecture:** `U-Net` with a pretrained `resnet34` backbone using `segmentation_models_pytorch`.
- **Loss Function:** Combination of **Dice Loss + Binary Cross Entropy (BCE)**.
- **Inference & Post-Processing:** Thresholding and Run-Length Encoding (RLE) to build `submission.csv`.

> *If you find this starter notebook helpful, please consider **upvoting**!* 👍


In [ ]:
import os
import sys
import json
import time
import warnings
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt

# Suppress Hugging Face & PyTorch warnings for clean output
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Install segmentation_models_pytorch if not present
try:
    import segmentation_models_pytorch as smp
except ImportError:
    print("📦 Installing segmentation-models-pytorch...")
    os.system("pip install -q segmentation-models-pytorch albumentations")
    import segmentation_models_pytorch as smp

import albumentations as A
from albumentations.pytorch import ToTensorV2

# Global Configuration
CFG = {
    'seed': 42,
    'img_size': (512, 512),
    'batch_size': 8,
    'epochs': 35,
    'lr': 3e-3,
    'backbone': 'resnet34',
    'encoder_weights': 'imagenet',
    'num_workers': 2,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'threshold': 0.5
}

def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['seed'])
print(f"🚀 Using Device: {CFG['device']}")


## 1. Paths & Annotation Helper Functions 📁¶
We set up paths and prepare a parser for the training annotations stored in MAGFiLO_1.0_Annotations_kaggle2026_train.json.



In [ ]:
BASE_DIR = Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/")
TRAIN_IMG_DIR = BASE_DIR / "train/train_images"
TEST_IMG_DIR = BASE_DIR / "test/test_images"
ANNOTATIONS_PATH = BASE_DIR / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"

# Load Annotations -- this file is itself COCO-style: {info, licenses, categories, images, annotations}.
# categories: 1=Left, 2=Right, 3=Unidentifiable, 4=Ambiguous filament.
with open(ANNOTATIONS_PATH, 'r') as f:
    coco_data = json.load(f)

CATEGORIES = {c['id']: c['name'] for c in coco_data['categories']}
NUM_CLASSES = len(CATEGORIES) + 1  # +1 for background, used by the Mask R-CNN head later

# Index annotations by image_id and image_id by file_name. The same underlying jpeg
# (file_name) can carry multiple independent annotator batches -- different "id" in
# coco_data['images'], same pixels -- so file_name -> image_id is one-to-many.
images_by_id = {im['id']: im for im in coco_data['images']}
anns_by_image_id = defaultdict(list)
for ann in coco_data['annotations']:
    anns_by_image_id[ann['image_id']].append(ann)

image_ids_by_file_name = defaultdict(list)
for im in coco_data['images']:
    image_ids_by_file_name[im['file_name']].append(im['id'])

# Get list of test images (no annotations available for these)
test_files = sorted([p.name for p in TEST_IMG_DIR.glob("*.jpeg")])
df_test = pd.DataFrame({'filename': test_files})

# Split by underlying file_name (NOT image_id) so that multiple annotator batches of the
# same photo always land on the same side of the split -- splitting by image_id would leak
# near-duplicate frames between train/val and inflate validation metrics.
file_names = sorted(image_ids_by_file_name.keys())
train_fns, val_fns = train_test_split(
    file_names,
    test_size=0.2,
    random_state=CFG['seed'],
    shuffle=True
)

def expand_to_image_ids(fns):
    ids = [im_id for fn in fns for im_id in image_ids_by_file_name[fn]]
    return pd.DataFrame({
        'image_id': ids,
        'filename': [images_by_id[i]['file_name'] for i in ids],
    }).reset_index(drop=True)

train_df = expand_to_image_ids(train_fns)
val_df = expand_to_image_ids(val_fns)

print(f"Total Train Images: {len(train_df)} (from {len(train_fns)} unique files) | "
      f"Validation Images: {len(val_df)} (from {len(val_fns)} unique files) | Test Images: {len(df_test)}")


## 2. Dataset & Mask Generation Pipeline 🧬¶
We construct a custom SolarDataset that loads .jpeg images and converts polygon/RLE coordinates from JSON into binary ground-truth masks (0 for background, 1 for filaments).



In [ ]:
from pycocotools import mask as mask_utils

def polygon_to_mask(segmentation, height, width):
    """Rasterize a COCO polygon segmentation ([[x0,y0,x1,y1,...], ...]) into a binary mask."""
    rles = mask_utils.frPyObjects(segmentation, height, width)
    rle = mask_utils.merge(rles) if len(rles) > 1 else rles[0]
    return mask_utils.decode(rle)

def build_binary_mask(image_id, height, width):
    """Union of every filament instance for one image_id into a single binary mask."""
    mask = np.zeros((height, width), dtype=np.uint8)
    for ann in anns_by_image_id.get(image_id, []):
        mask = np.maximum(mask, polygon_to_mask(ann['segmentation'], height, width))
    return mask

class SolarDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = row['filename']
        img_path = os.path.join(self.img_dir, filename)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, _ = image.shape

        if self.is_test:
            if self.transform:
                augmented = self.transform(image=image)
                image = augmented['image']
            return image, filename

        # Get target mask (binary union of all filament instances for this image_id)
        mask = build_binary_mask(row['image_id'], h, w)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        mask = mask.unsqueeze(0).float()
        return image, mask

# Augmentations using Albumentations
train_transform = A.Compose([
    A.Resize(CFG['img_size'][0], CFG['img_size'][1]),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(CFG['img_size'][0], CFG['img_size'][1]),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])


## 3. Model Architecture & Loss Function 🏗️¶
We utilize U-Net from segmentation_models_pytorch with an ImageNet pretrained ResNet-34 encoder. Our loss is a combination of Dice Loss (great for handling class imbalance) and Binary Cross Entropy (BCE).



In [4]:
# Create U-Net Model
def build_model():
    model = smp.Unet(
        encoder_name=CFG['backbone'],
        encoder_weights=CFG['encoder_weights'],
        in_channels=3,
        classes=1,
        activation=None  # Output logits
    )
    return model

# Hybrid Loss Function: Dice Loss + BCE Loss
class DiceBCELoss(nn.Module):
    def __init__(self):
        super(DiceBCELoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode='binary', from_logits=True)

    def forward(self, y_pred, y_true):
        return 0.5 * self.bce(y_pred, y_true) + 0.5 * self.dice(y_pred, y_true)

model = build_model().to(CFG['device'])
criterion = DiceBCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'])

print(f"✅ Model built with backbone '{CFG['backbone']}'")


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

✅ Model built with backbone 'resnet34'


## 4. Training Loop 🔁¶
Let's train the U-Net model across configured epochs while tracking train and validation loss alongside Dice metric scores.



In [ ]:
# Create DataLoaders
train_dataset = SolarDataset(train_df, TRAIN_IMG_DIR, transform=train_transform)
val_dataset = SolarDataset(val_df, TRAIN_IMG_DIR, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'])
val_loader = DataLoader(val_dataset, batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'])

best_val_loss = float('inf')

for epoch in range(CFG['epochs']):
    # Training Phase
    model.train()
    train_loss = 0.0
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{CFG['epochs']} [Train]"):
        images, masks = images.to(CFG['device']), masks.to(CFG['device'])
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    train_loss /= len(train_loader)
    
    # Validation Phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{CFG['epochs']} [Val]"):
            images, masks = images.to(CFG['device']), masks.to(CFG['device'])
            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item()
            
    val_loss /= len(val_loader)
    
    print(f"🌟 Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_unet_model.pth")
        print("   💾 Best model saved!")


## 5. RLE Encoding & Submission Generation 📄¶
Most Kaggle image segmentation competitions require predictions to be formatted as Run-Length Encoding (RLE) strings in submission.csv. Below is a standard, robust RLE encoding function.



In [ ]:
import pycocotools.mask as mask_utils

def mask_to_coco_rle(binary_mask):
    """
    Encodes a binary numpy mask (H, W) into COCO compressed RLE ASCII string.
    """
    # pycocotools expects Fortran-contiguous array (column-major)
    fortran_mask = np.asfortranarray(binary_mask.astype(np.uint8))
    rle = mask_utils.encode(fortran_mask)
    # Convert bytes to string for CSV
    rle['counts'] = rle['counts'].decode('utf-8')
    return rle['counts']

# Load best trained model weights
model.load_state_dict(torch.load("best_unet_model.pth"))
model.eval()

test_dataset = SolarDataset(df_test, TEST_IMG_DIR, transform=val_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

submissions = []

print("🔮 Generating Compressed COCO RLE Predictions for Test Images...")

with torch.no_grad():
    for images, filenames in tqdm(test_loader):
        images = images.to(CFG['device'])
        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy()[0, 0]

        # Predictions come out at CFG['img_size'] (512x512) because we resized on the way
        # in -- resize back up to the original image resolution before encoding RLE, since
        # that's the resolution submission masks are expected in.
        orig_h, orig_w = cv2.imread(os.path.join(TEST_IMG_DIR, filenames[0])).shape[:2]
        probs = cv2.resize(probs, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)

        # Binary thresholding
        binary_mask = (probs > CFG['threshold']).astype(np.uint8)
        
        image_name = Path(filenames[0]).stem  # e.g., '20110120105534Ch'
        
        # Find individual connected filament components
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask)
        
        filament_count = 0
        for label in range(1, num_labels):  # Skip background (label 0)
            # Filter out tiny noise artifacts
            if stats[label, cv2.CC_STAT_AREA] < 30: 
                continue
                
            single_filament_mask = (labels == label).astype(np.uint8)
            
            # Encode into COCO ASCII RLE format
            coco_rle_str = mask_to_coco_rle(single_filament_mask)
            
            if coco_rle_str:
                filament_count += 1
                submissions.append({
                    'filament_id': f"{image_name}_{filament_count}",
                    'segmentation_rle': coco_rle_str
                })
        
        # Fallback: If no filaments detected, encode an empty background mask
        if filament_count == 0:
            empty_mask = np.zeros(binary_mask.shape, dtype=np.uint8)
            empty_rle_str = mask_to_coco_rle(empty_mask)
            submissions.append({
                'filament_id': f"{image_name}_1",
                'segmentation_rle': empty_rle_str
            })

# Construct Submission DataFrame
sub_df = pd.DataFrame(submissions)

# Verify Output
print("\n🔍 Verification Check for NULL / NaN values:")
print(sub_df.isnull().sum())

print(f"\n📊 Total predictions generated: {len(sub_df)}")

# Save to CSV
sub_df.to_csv("submission.csv", index=False)
print("🎉 submission.csv generated successfully with COCO Compressed RLE strings!")
print(sub_df.head(10))


## 6. Alternative Model: Mask R-CNN (Instance Segmentation) 🎭¶

The U-Net above is a *binary semantic* segmentation model: it predicts one filament-vs-background
mask per image, and individual filaments are only recovered afterwards by running
`cv2.connectedComponentsWithStats` on that mask. That hack merges any filaments that touch or
overlap into a single instance, and discards the 4 annotated categories entirely
(`Left` / `Right` / `Unidentifiable` / `Ambiguous`).

The annotations are natively multi-instance (avg. ~7 filaments/image, up to 26) and multi-class,
which is exactly what **Mask R-CNN** predicts directly -- a set of (box, class, mask) triples per
image, with no post-hoc splitting needed. This section adds it as a second, independent pipeline
so you can compare both on the leaderboard; it does not touch the U-Net cells above.

We use `torchvision`'s `maskrcnn_resnet50_fpn` (COCO-pretrained) since it needs no extra installs
on Kaggle.


In [ ]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

MRCNN_CFG = {
    'img_size': 512,
    'batch_size': 2,
    'epochs': 10,
    'lr': 5e-4,
    'score_threshold': 0.5,
    'mask_threshold': 0.5,
}

mrcnn_train_transform = A.Compose(
    [
        A.Resize(MRCNN_CFG['img_size'], MRCNN_CFG['img_size']),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(format='coco', label_fields=['labels'], min_visibility=0.2),
)

mrcnn_val_transform = A.Compose(
    [
        A.Resize(MRCNN_CFG['img_size'], MRCNN_CFG['img_size']),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(format='coco', label_fields=['labels'], min_visibility=0.2),
)


class SolarInstanceDataset(Dataset):
    """Per-instance dataset for Mask R-CNN: one box/mask/label per filament (not merged)."""

    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(os.path.join(self.img_dir, row['filename']))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, _ = image.shape

        masks, boxes, labels = [], [], []
        for ann in anns_by_image_id.get(row['image_id'], []):
            m = polygon_to_mask(ann['segmentation'], h, w)
            if m.sum() == 0:
                continue
            ys, xs = np.where(m)
            boxes.append([float(xs.min()), float(ys.min()),
                          float(xs.max() - xs.min()), float(ys.max() - ys.min())])  # coco xywh
            masks.append(m)
            labels.append(ann['category_id'])

        augmented = self.transform(image=image, masks=masks, bboxes=boxes, labels=labels)
        image = augmented['image']
        aug_masks, aug_boxes, aug_labels = augmented['masks'], augmented['bboxes'], augmented['labels']

        size = MRCNN_CFG['img_size']
        if len(aug_boxes) == 0:
            boxes_t = torch.zeros((0, 4), dtype=torch.float32)
            masks_t = torch.zeros((0, size, size), dtype=torch.uint8)
            labels_t = torch.zeros((0,), dtype=torch.int64)
        else:
            xyxy = np.array([[x, y, x + bw, y + bh] for x, y, bw, bh in aug_boxes], dtype=np.float32)
            # torchvision's detection API hard-errors on degenerate (zero-area) boxes, which
            # augmentation can produce when an instance gets clipped to the resized frame edge.
            valid = (xyxy[:, 2] > xyxy[:, 0]) & (xyxy[:, 3] > xyxy[:, 1])
            if valid.sum() == 0:
                boxes_t = torch.zeros((0, 4), dtype=torch.float32)
                masks_t = torch.zeros((0, size, size), dtype=torch.uint8)
                labels_t = torch.zeros((0,), dtype=torch.int64)
            else:
                boxes_t = torch.as_tensor(xyxy[valid], dtype=torch.float32)
                masks_t = torch.as_tensor(np.stack(aug_masks)[valid], dtype=torch.uint8)
                labels_t = torch.as_tensor(np.array(aug_labels)[valid], dtype=torch.int64)

        target = {
            'boxes': boxes_t,
            'labels': labels_t,
            'masks': masks_t,
            'image_id': torch.tensor([idx]),
        }
        return image, target


def mrcnn_collate_fn(batch):
    return tuple(zip(*batch))


mrcnn_train_dataset = SolarInstanceDataset(train_df, TRAIN_IMG_DIR, transform=mrcnn_train_transform)
mrcnn_val_dataset = SolarInstanceDataset(val_df, TRAIN_IMG_DIR, transform=mrcnn_val_transform)

mrcnn_train_loader = DataLoader(
    mrcnn_train_dataset, batch_size=MRCNN_CFG['batch_size'], shuffle=True,
    num_workers=CFG['num_workers'], collate_fn=mrcnn_collate_fn,
)
mrcnn_val_loader = DataLoader(
    mrcnn_val_dataset, batch_size=MRCNN_CFG['batch_size'], shuffle=False,
    num_workers=CFG['num_workers'], collate_fn=mrcnn_collate_fn,
)

print(f"✅ Mask R-CNN datasets ready | Train: {len(mrcnn_train_dataset)} | Val: {len(mrcnn_val_dataset)}")


In [ ]:
def build_maskrcnn_model(num_classes):
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)
    return model

# NUM_CLASSES = background + 4 filament categories (Left, Right, Unidentifiable, Ambiguous)
mrcnn_model = build_maskrcnn_model(NUM_CLASSES).to(CFG['device'])
mrcnn_optimizer = torch.optim.AdamW(
    [p for p in mrcnn_model.parameters() if p.requires_grad], lr=MRCNN_CFG['lr']
)

print(f"✅ Mask R-CNN model built for {NUM_CLASSES} classes (background + {NUM_CLASSES - 1} filament types)")


In [ ]:
best_mrcnn_val_loss = float('inf')

for epoch in range(MRCNN_CFG['epochs']):
    # Training Phase
    mrcnn_model.train()
    train_loss = 0.0
    for images, targets in tqdm(mrcnn_train_loader, desc=f"Epoch {epoch+1}/{MRCNN_CFG['epochs']} [Train]"):
        images = [img.to(CFG['device']) for img in images]
        targets = [{k: v.to(CFG['device']) for k, v in t.items()} for t in targets]

        loss_dict = mrcnn_model(images, targets)
        loss = sum(loss_dict.values())

        mrcnn_optimizer.zero_grad()
        loss.backward()
        mrcnn_optimizer.step()

        train_loss += loss.item()

    train_loss /= len(mrcnn_train_loader)

    # Validation Phase -- torchvision detection models only return the loss dict in train()
    # mode (eval() mode returns predictions instead), so we keep train() but disable grad.
    mrcnn_model.train()
    val_loss = 0.0
    with torch.no_grad():
        for images, targets in tqdm(mrcnn_val_loader, desc=f"Epoch {epoch+1}/{MRCNN_CFG['epochs']} [Val]"):
            images = [img.to(CFG['device']) for img in images]
            targets = [{k: v.to(CFG['device']) for k, v in t.items()} for t in targets]
            loss_dict = mrcnn_model(images, targets)
            val_loss += sum(loss_dict.values()).item()

    val_loss /= len(mrcnn_val_loader)

    print(f"🎭 Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_mrcnn_val_loss:
        best_mrcnn_val_loss = val_loss
        torch.save(mrcnn_model.state_dict(), "best_maskrcnn_model.pth")
        print("   💾 Best Mask R-CNN model saved!")


In [ ]:
# Load best trained Mask R-CNN weights
mrcnn_model.load_state_dict(torch.load("best_maskrcnn_model.pth"))
mrcnn_model.eval()

mrcnn_test_transform = A.Compose([
    A.Resize(MRCNN_CFG['img_size'], MRCNN_CFG['img_size']),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

mrcnn_submissions = []

print("🎭 Generating Mask R-CNN instance predictions for Test Images...")

with torch.no_grad():
    for filename in tqdm(df_test['filename']):
        orig_image = cv2.imread(os.path.join(TEST_IMG_DIR, filename))
        orig_image = cv2.cvtColor(orig_image, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = orig_image.shape[:2]

        image_tensor = mrcnn_test_transform(image=orig_image)['image'].to(CFG['device'])
        pred = mrcnn_model([image_tensor])[0]  # detection API expects a list of images, not a batch dim

        image_name = Path(filename).stem
        keep = pred['scores'] > MRCNN_CFG['score_threshold']
        pred_masks = pred['masks'][keep].cpu().numpy()  # (N, 1, size, size) soft probability maps

        filament_count = 0
        for m in pred_masks:
            mask_pred = (m[0] > MRCNN_CFG['mask_threshold']).astype(np.uint8)
            # Resize each instance mask back up to the original image resolution before RLE encoding
            mask_orig = cv2.resize(mask_pred, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
            if mask_orig.sum() < 30:
                continue
            filament_count += 1
            mrcnn_submissions.append({
                'filament_id': f"{image_name}_{filament_count}",
                'segmentation_rle': mask_to_coco_rle(mask_orig)
            })

        # Fallback: If no filaments detected, encode an empty background mask
        if filament_count == 0:
            empty_mask = np.zeros((orig_h, orig_w), dtype=np.uint8)
            mrcnn_submissions.append({
                'filament_id': f"{image_name}_1",
                'segmentation_rle': mask_to_coco_rle(empty_mask)
            })

mrcnn_sub_df = pd.DataFrame(mrcnn_submissions)

print("\n🔍 Verification Check for NULL / NaN values:")
print(mrcnn_sub_df.isnull().sum())

print(f"\n📊 Total predictions generated: {len(mrcnn_sub_df)}")

# Saved under a separate filename so it doesn't clobber the U-Net's submission.csv --
# rename whichever pipeline's output you want to actually submit to "submission.csv".
mrcnn_sub_df.to_csv("submission_maskrcnn.csv", index=False)
print("🎉 submission_maskrcnn.csv generated successfully with COCO Compressed RLE strings!")
print(mrcnn_sub_df.head(10))
